# 04 — Optimisation de top_k
**RQ3** : Quel top_k offre le meilleur compromis entre Precision et Recall ?

Courbe Precision-Recall@k pour k = 1..15.

In [ ]:
import sys, os, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from config import load_config, RetrievalConfig
from retriever import retrieve_documents
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric

from notebooks.lib.reporter import load_benchmark
from notebooks.lib.plotter import comparison_curve
from experiments.registry import ExperimentLog

In [ ]:
benchmark = load_benchmark()
cfg = load_config()
judge = create_judge(cfg.evaluation)

TOP_K_VALUES = list(range(1, 16))

precision_metric = ContextualPrecisionMetric(threshold=0.75, model=judge, include_reason=True)
recall_metric = ContextualRecallMetric(threshold=0.75, model=judge, include_reason=True)
precision_metric.required_multiple = False
recall_metric.required_multiple = False

log = ExperimentLog(name="topk_optimisation")
log.set_params(top_k_values=TOP_K_VALUES)

results_by_k = {}

for k in TOP_K_VALUES:
    retrieval_cfg = RetrievalConfig(top_k=k, max_distance=cfg.retrieval.max_distance)
    precisions, recalls = [], []

    for idx, row in benchmark.iterrows():
        docs, scores = retrieve_documents(row['Question'], retrieval_cfg)
        if not docs:
            continue

        contexts = [d.page_content for d in docs]
        tc = LLMTestCase(
            input=row['Question'], actual_output="",
            expected_output=row['Ground_Truth'],
            retrieval_context=contexts,
        )

        try:
            precision_metric.measure(tc)
            precisions.append(precision_metric.score)
        except: precisions.append(0.0)

        try:
            recall_metric.measure(tc)
            recalls.append(recall_metric.score)
        except: recalls.append(0.0)

    results_by_k[k] = {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
    }

    log.record(top_k=k,
               Contextual_Precision=round(np.mean(precisions), 4),
               Contextual_Recall=round(np.mean(recalls), 4))
    print(f"top_k={k}: Precision={np.mean(precisions):.3f}, Recall={np.mean(recalls):.3f}")

In [ ]:
# Courbe Precision-Recall@k
precisions = [results_by_k[k]['precision'] for k in TOP_K_VALUES]
recalls = [results_by_k[k]['recall'] for k in TOP_K_VALUES]

comparison_curve(
    {"Contextual Precision": precisions, "Contextual Recall": recalls},
    title="Évolution des métriques selon top_k",
    filename='topk_precision_recall_curve.png',
)

# Identifier le point optimal (F1 max)
f1_scores = [
    2 * (p * r) / (p + r) if (p + r) > 0 else 0
    for p, r in zip(precisions, recalls)
]
optimal_k = TOP_K_VALUES[np.argmax(f1_scores)]
print(f"\nTop_k optimal (F1 max) : k={optimal_k} (F1={max(f1_scores):.3f})")

log.save_all()
log.append_to_global_log()